# Model & data audit — Cordilla account scoring

## Objective

Determine whether `model/model.pkl` can be trusted to influence how reps allocate their limited attention, and whether it's worth investing further effort in it — not to retrain or improve the model, but to decide what to trust it for, what not to, and whether to ship its scores as-is.

## Audit questions — as written before starting

This is the list the audit set out with, left unedited. It did not get through it: attempting Q1 revealed there is no data to validate performance against, and what that exposed about the training data was more decision-relevant than the remaining questions. See **Check 2** for where it stopped and why.

1. **Performance, overall and by segment.** How good is the model in aggregate, and does that hold evenly across account segments — or is it materially more (or less) trustworthy for some segments than others?
2. **Variable correlation to target.** Which features actually move with `converted_within_90d`, and do the relationships make sense?
3. **Is the model still useful as-is?** How old is the training data, and how does its distribution compare to `accounts_to_score.csv`? Data drift is the leading suspect for what likely killed the earlier Cordilla scoring effort, so this checks whether the same failure mode applies here.
4. **Label leakage / circularity.** Are any features (e.g. `sales_contacts_90d`, marketing engagement fields) consequences of a rep already having contacted the account, rather than independent predictors available *before* that contact decision? If so the model may be learning "who did we already talk to," not "who is likely to convert" — which would make it unusable for the attention-allocation decision it's meant to support.
5. **Selection bias in the training population.** Was `training_data.csv` a representative sample of the account universe, or already filtered by whatever selection reps or managers applied historically? If the latter, the model never learned from the accounts that were ignored, and its scores may not generalize to the full `accounts_to_score.csv` universe.
6. **Baseline comparison.** Does the model meaningfully outperform a trivial heuristic (e.g., sorting by company size or an existing engagement field)? If a simple rule captures most of the value, that changes the recommendation.
7. **Feature parity between training and scoring data.** Do all training features exist in `accounts_to_score.csv`, with the same definitions, units, and missingness patterns? A silently mismatched or missing column would break the model quietly.

## Noted limitation (not checked here)

`converted_within_90d` is a conversion count/flag, not a revenue figure — if deal sizes vary a lot, a model optimized for conversion probability could systematically undervalue high-ARR accounts. There's no deal-size field available to verify this either way, so it's flagged as an open limitation rather than something this audit can resolve.


## Check 0 — Load and inspect the data

**Purpose:** before testing any hypothesis, confirm what we actually have — shapes, columns, dtypes, and a few raw rows from both `training_data.csv` and `accounts_to_score.csv`.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

TODAY = pd.Timestamp("2026-08-01")  # per CLAUDE.md: treat this as "today", not the system clock

train = pd.read_csv("../data/training_data.csv")
score = pd.read_csv("../data/accounts_to_score.csv")

print("training_data.csv:", train.shape)
print("accounts_to_score.csv:", score.shape)

training_data.csv: (1200, 12)
accounts_to_score.csv: (300, 11)


In [2]:
train.head()

,account_id,account_type,snapshot_date,employee_count,industry,intent_score,mql_count_90d,trial_started,trial_active_users,web_touchpoints_90d,sales_contacts_90d,converted_within_90d
0,ACC-00002,Suspect,2026-07-07,21,Retail,NaN,2,1,1,9,4,0
1,ACC-00003,Suspect,2024-11-27,21,Professional Services,30.9,0,1,1,8,0,0
2,ACC-00004,Prospect,2025-01-18,72,Financial Services,NaN,4,0,0,2,0,0
3,ACC-00005,Prospect,2026-06-22,54,Manufacturing,22.7,0,0,0,3,0,0
4,ACC-00006,Former Customer,2026-01-26,63,Software,NaN,2,0,0,8,0,0


In [3]:
score.head()

,account_id,account_type,snapshot_date,employee_count,industry,intent_score,mql_count_90d,trial_started,trial_active_users,web_touchpoints_90d,sales_contacts_90d
0,ACC-01073,Prospect,2026-04-11,9,Healthcare,NaN,1,0,0,2,0
1,ACC-00533,Prospect,2026-07-19,65,Manufacturing,63.0,1,0,0,7,0
2,ACC-01319,Prospect,2025-08-15,17,Retail,NaN,0,0,0,5,0
3,ACC-00160,Suspect,2026-04-06,63,Professional Services,6.9,0,0,0,8,3
4,ACC-00275,Suspect,2026-04-05,80,Healthcare,46.3,0,0,0,0,1


In [4]:
train.dtypes

account_id               object
account_type             object
snapshot_date            object
employee_count            int64
industry                 object
intent_score            float64
mql_count_90d             int64
trial_started             int64
trial_active_users        int64
web_touchpoints_90d       int64
sales_contacts_90d        int64
converted_within_90d      int64
dtype: object

## Check 1 — Target variable balance

**Hypothesis:** 1,200 total rows may sound adequate, but if `converted_within_90d` is a rare event, the number of *positive* examples the model actually learned from could be small enough to make its patterns fragile — especially once we look at segments or correlations later.

**Test:** count and rate of `converted_within_90d` in `training_data.csv`, overall and by `account_type`.

In [5]:
n_pos = train["converted_within_90d"].sum()
n_total = len(train)
print(f"Positives: {n_pos} / {n_total} ({n_pos / n_total:.1%})")

by_type = train.groupby("account_type")["converted_within_90d"].agg(
    accounts="count", conversions="sum"
)
by_type["% of dataset"] = (by_type["accounts"] / n_total * 100).round(1)
by_type["conversion rate %"] = (by_type["conversions"] / by_type["accounts"] * 100).round(1)
by_type["% of all conversions"] = (by_type["conversions"] / n_pos * 100).round(1)

print("\nBy account_type:")
by_type[["accounts", "% of dataset", "conversions", "conversion rate %", "% of all conversions"]]

Positives: 78 / 1200 (6.5%)

By account_type:


,accounts,% of dataset,conversions,conversion rate %,% of all conversions
account_type,,,,,
Former Customer,166,13.8,12,7.2,15.4
Prospect,632,52.7,42,6.6,53.8
Suspect,402,33.5,24,6.0,30.8


## Check 2 — model performance cannot be validated

Setting out to answer Q1, we hit a wall immediately: there is nothing to test against. The brief states that `training_data.csv` is the data the model was trained on, and `accounts_to_score.csv` is unlabeled.

> **The available data does not allow us to independently validate out-of-sample model performance, because no labeled holdout set is provided.**

Anything computed against `training_data.csv` is the model graded on data it has already seen, and overstates real performance by an unknown margin. We scored it anyway before stopping — ~0.76 AUC, ~4× lift in the top decile — and removed those numbers rather than publish them with a caveat, since the caveat does not survive being quoted.

**Ask:** a properly separated labeled holdout set is required before any performance claim about this model can be made. That is a blocking dependency, not a nice-to-have.

**Where the audit goes instead.** A performance number would not have settled this anyway — a test set drawn from this same data inherits whatever is wrong with the data. Defects in the training material sit upstream of any metric, so the remaining checks go after the data and how the model was fit to it.

## Check 3 — Are the labels observable? (censored outcomes)

**Hypothesis:** `converted_within_90d` needs 90 days to resolve. With 2026-08-01 as "today," any account snapshotted after 2026-05-03 has not had that window close, so its label cannot be a real outcome. If those rows are recorded as `0`, the model was taught that accounts failed when the answer simply wasn't in yet.

**Test:** conversion rate by snapshot age. If censoring is real, the youngest bucket will show an impossible zero rather than a gradual decline.

In [6]:
train["snapshot_date"] = pd.to_datetime(train["snapshot_date"])
train["age_days"] = (TODAY - train["snapshot_date"]).dt.days

buckets = pd.cut(train["age_days"], [0, 90, 180, 365, 547, 730],
                 labels=["0-90d (window still open)", "90-180d", "180-365d", "1-1.5y", "1.5-2y"])

print("Conversion rate by snapshot age:")
print(train.groupby(buckets, observed=True)["converted_within_90d"]
           .agg(accounts="count", conversions="sum", rate="mean").round(4))

censored = train["age_days"] <= 90
print(f"\nAccounts whose 90-day window had not closed: {censored.sum()} ({censored.mean():.1%} of training data)")
print(f"Conversions among them: {train.loc[censored, 'converted_within_90d'].sum()}")
print(f"\nBase rate including these rows: {train['converted_within_90d'].mean():.3%}")
print(f"Base rate excluding these rows: {train.loc[~censored, 'converted_within_90d'].mean():.3%}")

Conversion rate by snapshot age:
                           accounts  conversions    rate
age_days                                                
0-90d (window still open)       103            0  0.0000
90-180d                         220           16  0.0727
180-365d                        484           39  0.0806
1-1.5y                          304           18  0.0592
1.5-2y                           89            5  0.0562

Accounts whose 90-day window had not closed: 103 (8.6% of training data)
Conversions among them: 0

Base rate including these rows: 6.500%
Base rate excluding these rows: 7.110%


## Check 4 — Is `intent_score` missingness informative?

**First, where the blanks are and where they aren't.** In `training_data.csv` itself, 482 rows are genuinely blank — the file is never modified, and that raw blankness is what's measured below. Inside the pipeline, `SimpleImputer` replaces every blank with the median (25.3). That happens at **fit time as well as predict time**: `Pipeline.fit()` runs `fit_transform` on the preprocessing steps before the classifier sees anything, and `GradientBoostingClassifier` rejects NaN outright. So the model was *trained* on imputed values and never had the opportunity to learn from missingness at all — it isn't only a scoring-path issue.

**Hypothesis (from the brief):** intent data is missing on ~40% of rows because vendor coverage skews toward larger accounts. If true, missingness is partly a proxy for company size.

**Test, three parts:**
- (a) Does *coverage* track account size, as the brief says?
- (b) Do the *score values themselves* track account size — the other reading of "skews toward larger accounts"?
- (c) Does missingness track conversion, and if so, is that just company size in disguise?

In [7]:
missing = train["intent_score"].isna()
size_q = pd.qcut(train["employee_count"], 4)

print(f"intent_score blank in the CSV: {missing.sum()} rows "
      f"({missing.mean():.1%} of training, {score['intent_score'].isna().mean():.1%} of scoring)\n")

# (a) does COVERAGE track account size, as the brief claims?
print("(a) Missing rate by company-size quartile:")
print(train.groupby(size_q, observed=True)
           .apply(lambda g: pd.Series({"accounts": len(g),
                                       "% missing": round(g["intent_score"].isna().mean() * 100, 1)}),
                  include_groups=False))

# (b) do the VALUES track account size — the other reading of "skews toward larger accounts"?
print("\n(b) intent_score value by company-size quartile (where present):")
print(train[~missing].groupby(size_q, observed=True)["intent_score"]
           .agg(accounts="count", mean="mean", median="median").round(2))
print(f"    correlation(employee_count, intent_score) = {train['intent_score'].corr(train['employee_count']):.3f}")

# (c) does missingness track conversion — and is that just size in disguise?
print("\n(c) Conversion rate by whether intent_score is present:")
print(train.groupby(missing)["converted_within_90d"]
           .agg(accounts="count", conversions="sum", rate="mean").round(4)
           .rename(index={False: "intent present", True: "intent MISSING"}))

print("\n    ...and the same split held inside each size quartile (controls for size):")
print(train.groupby([size_q, missing], observed=True)["converted_within_90d"]
           .agg(accounts="count", rate="mean").round(3))

# Confirm the model was TRAINED on imputed values, not just scored with them
from sklearn.ensemble import GradientBoostingClassifier
try:
    GradientBoostingClassifier(n_estimators=2).fit(np.array([[1.0], [2.0], [np.nan], [4.0]]), [0, 1, 0, 1])
    accepts_nan = True
except ValueError:
    accepts_nan = False

print(f"\nValue substituted for every blank (median): {train['intent_score'].median()}")
print(f"Classifier tolerates NaN: {accepts_nan}  -> it never saw a blank, at fit time or predict time")

intent_score blank in the CSV: 482 rows (40.2% of training, 38.7% of scoring)

(a) Missing rate by company-size quartile:
                 accounts  % missing
employee_count                      
(2.999, 29.0]       304.0       40.8
(29.0, 67.0]        304.0       42.8
(67.0, 139.0]       295.0       38.6
(139.0, 4429.0]     297.0       38.4

(b) intent_score value by company-size quartile (where present):
                 accounts   mean  median
employee_count                          
(2.999, 29.0]         180  27.48    24.1
(29.0, 67.0]          174  27.69    25.5
(67.0, 139.0]         181  28.13    27.3
(139.0, 4429.0]       183  28.07    25.4
    correlation(employee_count, intent_score) = 0.039

(c) Conversion rate by whether intent_score is present:
                accounts  conversions    rate
intent_score                                 
intent present       718           59  0.0822
intent MISSING       482           19  0.0394

    ...and the same split held inside each size 


Value substituted for every blank (median): 25.3
Classifier tolerates NaN: False  -> it never saw a blank, at fit time or predict time


## Check 5 — How stale is the training data, and does it match what we're scoring?

**Hypothesis:** the model has no `snapshot_date` feature (confirmed in Check 2 — it takes nine inputs, none of them the date). So it treats a two-year-old snapshot and a fresh one as equally valid descriptions of an account. If the training data is systematically older than the accounts we're about to score, the model is applying yesterday's patterns to today's accounts with no way to know it.

**Test:** snapshot age distribution in both files, plus a side-by-side of feature means to see whether the two populations otherwise look alike.

In [8]:
score["snapshot_date"] = pd.to_datetime(score["snapshot_date"])
score["age_days"] = (TODAY - score["snapshot_date"]).dt.days

print("Snapshot age in days (today = 2026-08-01):")
print(pd.DataFrame({
    "training data": train["age_days"].describe()[["min", "50%", "mean", "max"]],
    "accounts to score": score["age_days"].describe()[["min", "50%", "mean", "max"]],
}).round(1))

NUM = ["employee_count", "intent_score", "mql_count_90d", "trial_started",
       "trial_active_users", "web_touchpoints_90d", "sales_contacts_90d"]
print("\nFeature means, training vs scoring:")
print(pd.DataFrame({"training": train[NUM].mean(), "scoring": score[NUM].mean()}).round(2))

print("\nAccount type mix:")
print(pd.DataFrame({"training": train["account_type"].value_counts(normalize=True),
                    "scoring": score["account_type"].value_counts(normalize=True)}).round(3))

Snapshot age in days (today = 2026-08-01):
      training data  accounts to score
min             1.0                0.0
50%           280.0              121.0
mean          296.8              183.8
max           711.0              675.0

Feature means, training vs scoring:
                     training  scoring
employee_count         121.14   107.75
intent_score            27.85    30.08
mql_count_90d            1.09     1.30
trial_started            0.19     0.17
trial_active_users       0.34     0.25
web_touchpoints_90d      3.08     2.99
sales_contacts_90d       1.60     1.60

Account type mix:
                 training  scoring
account_type                      
Prospect            0.527    0.543
Suspect             0.335    0.337
Former Customer     0.138    0.120


## Check 6 — Integrity sweep, and does the model just recognise accounts reps already called?

**Hypothesis:** `sales_contacts_90d` counts rep outreach in the same 90-day window the outcome is measured over. If conversion rises with contacts, the model may partly be learning "someone already worked this account" — which is circular for a tool meant to decide who to work *next*.

**Also swept here:** duplicate rows, train/score account overlap, impossible values, and feature parity — the boring checks that are worth ruling out explicitly rather than assuming.

In [9]:
print("Conversion rate by number of rep contacts in the same 90-day window:")
print(train.groupby("sales_contacts_90d")["converted_within_90d"]
           .agg(accounts="count", conversions="sum", rate="mean").round(3))

print("\n--- integrity sweep ---")
print("duplicate account_ids in training :", train["account_id"].duplicated().sum())
print("fully duplicated rows             :", train.duplicated().sum())
print("accounts in both train and score  :", len(set(train["account_id"]) & set(score["account_id"])))
print("trial_active_users > 0 but trial_started == 0:",
      ((train["trial_active_users"] > 0) & (train["trial_started"] == 0)).sum())
print("negative values anywhere          :",
      int((train[NUM] < 0).sum().sum()))
print("columns in train but not score    :", set(train.columns) - set(score.columns) - {"score", "age_days"})
print("industry categories identical     :",
      sorted(train["industry"].unique()) == sorted(score["industry"].unique()))
print("account_type categories identical :",
      sorted(train["account_type"].unique()) == sorted(score["account_type"].unique()))

Conversion rate by number of rep contacts in the same 90-day window:
                    accounts  conversions   rate
sales_contacts_90d                              
0                        487           23  0.047
1                        188           13  0.069
2                        176           11  0.062
3                        159            9  0.057
4                        100           10  0.100
5                         59            8  0.136
6                         18            1  0.056
7                         10            2  0.200
8                          2            0  0.000
9                          1            1  1.000

--- integrity sweep ---
duplicate account_ids in training : 0
fully duplicated rows             : 0
accounts in both train and score  : 0
trial_active_users > 0 but trial_started == 0: 0
negative values anywhere          : 0
columns in train but not score    : {'converted_within_90d'}
industry categories identical     : True
account_type ca

## Check 7 — Why does this data disagree with the brief: bad sampling, or a shifted world?

The brief describes the real environment: conversion "well under 1% for cold accounts and low single digits for anything with recent engagement," an account universe of "tens of thousands of noncustomers, mostly untouched," and intent coverage skewed to larger accounts. This dataset matches none of that — it converts at 6.5% and shows flat intent coverage.

**Two explanations, and they lead to different decisions:**
- **Selection** — the sample was drawn badly, over-representing accounts that were worked and/or converted. Then the model learned relationships from a slice of the world, and we cannot know which hold outside it.
- **Drift** — the world genuinely changed between when this data was collected and now. Then the data was fine when built, and the fix is retraining on fresh data.

**Test:** if the world had shifted, the change should be visible *within* the two years this data spans — conversion rates and vendor coverage moving over time. If it's selection, the data will look internally stable but sit at the wrong level, and the sample will look unlike the described universe.

In [10]:
resolved = train[train["age_days"] > 90]  # drop censored rows so labels are real

# 1. Does this sample look like the "mostly untouched" universe the brief describes?
print("Share of accounts with NO rep contact  :", round((train["sales_contacts_90d"] == 0).mean(), 3))
print("Share already contacted by a rep       :", round((train["sales_contacts_90d"] > 0).mean(), 3))

# 2. Even the genuinely cold accounts — do they convert at the brief's <1%?
cold = ((resolved["sales_contacts_90d"] == 0) & (resolved["mql_count_90d"] == 0)
        & (resolved["trial_started"] == 0) & (resolved["web_touchpoints_90d"] <= 1))
print(f"\nGenuinely cold accounts (no contact, no MQL, no trial, <=1 web touch): {cold.sum()}")
print(f"  their conversion rate : {resolved.loc[cold, 'converted_within_90d'].mean():.2%}  (brief says <1%)")
print(f"  everyone else         : {resolved.loc[~cold, 'converted_within_90d'].mean():.2%}")

# 3. Is there a time trend? (drift would show up here)
age_buckets = pd.cut(resolved["age_days"], [90, 180, 365, 547, 730],
                     labels=["90-180d", "180-365d", "1-1.5y", "1.5-2y"])
print("\nConversion rate over time (drift would show as a trend):")
print(resolved.groupby(age_buckets, observed=True)["converted_within_90d"]
              .agg(accounts="count", rate="mean").round(4))

print("\nIntent coverage over time (a vendor policy change would show here):")
print(resolved.groupby(age_buckets, observed=True)["intent_score"]
              .apply(lambda s: round(s.isna().mean(), 3)).rename("% missing"))

Share of accounts with NO rep contact  : 0.406
Share already contacted by a rep       : 0.594

Genuinely cold accounts (no contact, no MQL, no trial, <=1 web touch): 71
  their conversion rate : 4.23%  (brief says <1%)
  everyone else         : 7.31%

Conversion rate over time (drift would show as a trend):
          accounts    rate
age_days                  
90-180d        220  0.0727
180-365d       484  0.0806
1-1.5y         304  0.0592
1.5-2y          89  0.0562

Intent coverage over time (a vendor policy change would show here):
age_days
90-180d     0.405
180-365d    0.401
1-1.5y      0.398
1.5-2y      0.360
Name: % missing, dtype: float64


## Check 8 — Which variables actually separate converters from non-converters?

**Question:** setting the model aside, which raw signals in this data carry information about conversion, and which are close to noise? This says what a rebuilt model would have to work with, and gives a sense of whether the useful signal is concentrated in one or two fields.

**Method:** conversion rate by bin for each variable, run on the **1,097 rows whose 90-day window had actually closed** — the 103 censored rows carry false zeros (Check 3) and would drag every rate down unevenly. Categorical and low-cardinality fields are shown as-is; continuous ones are split into quartiles.

**Read the counts, not just the rates.** With 78 conversions total, a bin holding 30 accounts and 4 conversions produces a rate that looks precise and isn't. The ranking at the end only considers bins with at least 50 accounts for that reason.

In [11]:
base = resolved["converted_within_90d"].mean()
print(f"Base conversion rate on resolved rows: {base:.2%}  ({len(resolved)} accounts, "
      f"{int(resolved['converted_within_90d'].sum())} conversions)\n")

COUNT_FIELDS = ["mql_count_90d", "trial_active_users", "web_touchpoints_90d", "sales_contacts_90d"]


def bin_variable(df, col):
    s = df[col]
    if s.dtype == object or col == "trial_started":
        return s
    if col == "intent_score":  # keep "no vendor coverage" as its own group
        return pd.qcut(s, 4).cat.add_categories(["NO COVERAGE"]).fillna("NO COVERAGE")
    if col in COUNT_FIELDS:    # counts: fixed buckets, long tail grouped
        return pd.cut(s, [-1, 0, 1, 3, s.max()], labels=["0", "1", "2-3", "4+"])
    return pd.qcut(s, 4)       # continuous: quartiles


VARIABLES = ["account_type", "industry", "employee_count", "intent_score",
             "mql_count_90d", "trial_started", "trial_active_users",
             "web_touchpoints_90d", "sales_contacts_90d"]

for col in VARIABLES:
    tbl = (resolved.groupby(bin_variable(resolved, col), observed=True)["converted_within_90d"]
                   .agg(accounts="count", conversions="sum", rate="mean"))
    tbl["rate %"] = (tbl["rate"] * 100).round(1)
    tbl["lift"] = (tbl["rate"] / base).round(2)
    print(f"--- {col} ---")
    print(tbl[["accounts", "conversions", "rate %", "lift"]].to_string())
    print()

Base conversion rate on resolved rows: 7.11%  (1097 accounts, 78 conversions)

--- account_type ---
                 accounts  conversions  rate %  lift
account_type                                        
Former Customer       150           12     8.0  1.13
Prospect              580           42     7.2  1.02
Suspect               367           24     6.5  0.92

--- industry ---
                       accounts  conversions  rate %  lift
industry                                                  
Financial Services          164           12     7.3  1.03
Healthcare                  177           13     7.3  1.03
Manufacturing               181           16     8.8  1.24
Professional Services       186           12     6.5  0.91
Retail                      191           11     5.8  0.81
Software                    198           14     7.1  0.99

--- employee_count ---
                 accounts  conversions  rate %  lift
employee_count                                      
(2.999, 29.0]  

In [12]:
rows = []
for col in VARIABLES:
    tbl = (resolved.groupby(bin_variable(resolved, col), observed=True)["converted_within_90d"]
                   .agg(n="count", k="sum", rate="mean"))
    tbl = tbl[tbl["n"] >= 50]                     # ignore bins too small to read
    if len(tbl) < 2:
        continue
    hi, lo = tbl["rate"].idxmax(), tbl["rate"].idxmin()
    p1, n1 = tbl.loc[hi, "rate"], tbl.loc[hi, "n"]
    p0, n0 = tbl.loc[lo, "rate"], tbl.loc[lo, "n"]
    # is the gap bigger than sampling noise? (2 standard errors on the difference)
    se = np.sqrt(p1 * (1 - p1) / n1 + p0 * (1 - p0) / n0)
    rows.append({
        "variable": col,
        "best bin": str(hi), "best %": round(p1 * 100, 1),
        "worst bin": str(lo), "worst %": round(p0 * 100, 1),
        "gap (pp)": round((p1 - p0) * 100, 1),
        "beats noise?": "yes" if (p1 - p0) > 2 * se else "no",
    })

summary = pd.DataFrame(rows).sort_values("gap (pp)", ascending=False).set_index("variable")
print("Variables ranked by how much they separate converters (resolved rows only):\n")
print(summary.to_string())
print("\n'beats noise?' = gap exceeds 2 standard errors. With only 78 conversions,")
print("anything marked 'no' is indistinguishable from chance and should not be read as signal.")

Variables ranked by how much they separate converters (resolved rows only):

                            best bin  best %     worst bin  worst %  gap (pp) beats noise?
variable                                                                                  
sales_contacts_90d                4+    12.5             0      5.3       7.2          yes
intent_score            (15.1, 25.1]    10.9   NO COVERAGE      4.4       6.6          yes
trial_active_users               2-3    11.3             0      6.7       4.6           no
trial_started                      1    10.8             0      6.3       4.5           no
mql_count_90d                      1     8.7            4+      4.3       4.4           no
web_touchpoints_90d               4+     8.0             1      3.8       4.2           no
employee_count         (2.999, 29.0]     8.3  (29.0, 66.0]      4.8       3.5           no
industry               Manufacturing     8.8        Retail      5.8       3.1           no
account_type 

### What Check 8 says

**Only two of the nine variables separate converters by more than sampling noise — and one of them is circular.**

- **`sales_contacts_90d`** is the widest gap (12.5% at 4+ contacts vs 5.3% at zero). But it records rep action taken during the same 90 days the outcome is measured, so it's partly "accounts someone already worked." It cannot be used to decide who to work *next* without assuming the answer.
- **`intent_score`** is the only clean signal that clears the bar — and the useful part is **whether the vendor has coverage at all**, not the number. Accounts with no coverage convert at 4.4%; accounts with coverage convert at 6.7–10.9% *with no ordering across the range* (7.2 → 10.9 → 6.7 → 10.9 across quartiles). The score's value is close to meaningless; its presence is the signal.

**Everything else is indistinguishable from chance at this sample size.** `trial_started` (10.8% vs 6.3%) and `trial_active_users` are directionally plausible and worth revisiting with more data, but with 204 and 109 accounts respectively the gaps don't clear two standard errors. `industry`, `account_type`, `employee_count`, `mql_count_90d` and `web_touchpoints_90d` show non-monotonic patterns — company size runs 8.3 / 4.8 / 8.0 / 7.3 across quartiles, web touchpoints 7.9 / 3.8 / 5.4 / 8.0 — which is what noise looks like, not a relationship.

**Why this matters for the decision.** The model takes nine inputs. On this data, one carries clean non-circular signal, one carries circular signal, and the rest are close to noise. And the single clean signal is the one the pipeline destroys by median-imputing before training (Check 4). That is a compact case that the problem is the training material rather than the algorithm: there is very little here for any model to learn from, and the most usable piece was discarded before learning started.

## Findings

**1. Performance cannot be validated at all.**
`training_data.csv` is what the model was trained on; `accounts_to_score.csv` is unlabeled. No holdout exists, so every metric is the model graded on its own homework. *A properly separated labeled test set is a blocking dependency before any performance claim.*

**2. 103 accounts are labeled "did not convert" when the outcome was unobservable.**
Conversion by snapshot age: 7.3% / 8.1% / 5.9% / 5.6% across older buckets — and **exactly 0.0% across the 103 accounts whose 90-day window hadn't closed** by 2026-08-01. A hard zero over 103 rows is mechanical, not behavioural. *8.6% of training rows carry a label that is false by construction; the true base rate is 7.11%, not the 6.50% the model was fit on.*

**3. The training sample isn't the population reps work — and it's selection, not drift.**
- 59.4% of these accounts have already been contacted by a rep, in a universe the brief describes as "mostly untouched."
- Even genuinely cold accounts (no contact, no MQL, no trial, ≤1 web touch) convert at 4.2%, against the brief's "well under 1%" — so enrichment runs through the sample, not just its mix. *(71 accounts; noisy.)*
- Drift is ruled out: across the two years the data spans, conversion is 7.3% / 8.1% / 5.9% / 5.6% and intent coverage 40.5% / 40.1% / 39.8% / 36.0% — no trend either way. The data is internally stable and simply sits at the wrong level.

*This cannot be fixed by rescaling the base rate. Selection is on features (contacted, engaged), not just the outcome, so an intercept correction wouldn't restore the ranking.*

**4. The model was never given the chance to learn its strongest available signal.**
- Accounts **with** intent data convert at 8.2%; **without**, 3.9% — over 2×, and it holds inside every size quartile (10.0/4.0, 6.9/1.5, 7.7/6.1, 8.2/4.4), so it isn't company size in disguise.
- All 482 blanks were replaced with the median (25.3) **before training, not just at scoring time**: `Pipeline.fit()` runs `fit_transform` on the preprocessing steps before the classifier sees anything, and `GradientBoostingClassifier` rejects NaN outright. The classifier has never encountered a missing intent value in its life. It cannot distinguish "middling intent" from "no vendor coverage," and never could. *A missing-indicator flag would have preserved it.*
- Note: the brief's stated reason for the gaps — coverage skewing to larger accounts — is **not true in this file**. Coverage by size quartile is 40.8/42.8/38.6/38.4%, and score values are flat too (correlation with employee count 0.039).

**5. The model is blind to how stale an account picture is.**
Training snapshots median **280 days** old (reaching back 711); accounts to be scored median **121 days**. `snapshot_date` is not among the model's nine inputs, so a two-year-old picture and last month's are treated identically. Every other distribution matches closely between the two files — this age gap is the substantive difference.

**6. Circularity is present but modest.**
Conversion rises with rep contacts (4.7% at zero, 10–14% at four or five), both measured over the same 90 days. Cause and selection can't be separated from this data. *A real caveat for a tool meant to choose who to contact next, but not large enough to explain the model on its own; cells above five contacts are too small to read.*

**7. Ruled out — the plumbing is clean.**
No duplicate account_ids or rows, no train/score overlap, no impossible trial values, no negatives, identical columns and category sets across both files. *The failures are in how the data was labeled and sampled, not how it was assembled.*

---

**Bottom line.** The model can't be validated, and the material it learned from is mislabeled on 8.6% of rows, drawn from the wrong population, stripped of its best signal before training even began, and roughly five months staler than what it would score. None of this is fixable by tuning — it requires rebuilding the training set. Its scores should not reach a rep as a number presented as trustworthy.